# Лабораторна робота №4: Інтерактивний додаток цифрової обробки сигналів
**Виконав:** Студент групи ФБ-46 — Ільченко Владислав
**Технологічний стек:** Python, NumPy, SciPy (signal.butter, signal.filtfilt), Блокнот Jupyter (ipywidgets)

### Крок 1. Імпорт модулів та ініціалізація графічного бекенду

In [9]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.signal import butter, filtfilt

print("=== СЕРЕДОВИЩЕ СИНХРОНІЗОВАНО УСПІШНО ===")

=== СЕРЕДОВИЩЕ СИНХРОНІЗОВАНО УСПІШНО ===


### Крок 2. Реалізація обов'язкової функції harmonic_with_noise згідно з ТЗ
*Функція генерує математичну модель гармонічного сигналу та накладає на нього випадковий гауссів шум за умови активного прапорця.*

In [10]:
def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    # Вісь часу робимо глобальною або локальною на 1000 точок
    t_local = np.linspace(0, 1, 1000)
    harmonic = amplitude * np.sin(2 * np.pi * frequency * t_local + phase)
    
    if show_noise:
        # Шум генерується на основі переданої дисперсії (коvariance)
        noise = np.random.normal(noise_mean, np.sqrt(noise_covariance), len(t_local))
        return harmonic, noise
    else:
        return harmonic, np.zeros(len(t_local))

print("=== ФУНКЦІЯ harmonic_with_noise РЕАЛІЗОВАНА ЗА ТЗ ===")

=== ФУНКЦІЯ harmonic_with_noise РЕАЛІЗОВАНА ЗА ТЗ ===


### Крок 3. Побудова інтерактивного GUI з кнопкою Reset та фільтром Баттерворта
*Кнопка Reset скидає параметри, а логіка відстеження змін гарантує, що шум не перегенеровується, якщо параметри шуму не змінювались.*

In [11]:
# 1. Глобальні координати часу та початковий фіксований шум
t = np.linspace(0, 1, 1000)
np.random.seed(42)
base_noise = np.random.normal(0, 1, len(t))

# Зберігаємо попередні значення шуму для перевірки умов ТЗ
prev_mean = 0.0
prev_cov = 0.1
current_noise = (base_noise * np.sqrt(prev_cov)) + prev_mean

# Початкові константи для кнопки Reset
INIT_AMP = 1.0
INIT_FREQ = 5.0
INIT_PHASE = 0.0
INIT_MEAN = 0.0
INIT_COV = 0.1
INIT_CUTOFF = 15.0

# 2. Оголошення віджетів (Слайдери, Чекбокс, Кнопка)
s_amp = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=INIT_AMP, description='Амплітуда (A)')
s_freq = widgets.FloatSlider(min=1.0, max=30.0, step=0.5, value=INIT_FREQ, description='Частота (f)')
s_phase = widgets.FloatSlider(min=0.0, max=2*np.pi, step=0.1, value=INIT_PHASE, description='Фаза (phi)')
s_mean = widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=INIT_MEAN, description='Шум (сер.)')
s_cov = widgets.FloatSlider(min=0.01, max=2.0, step=0.05, value=INIT_COV, description='Шум (дисп.)')
s_cutoff = widgets.FloatSlider(min=1.0, max=50.0, step=0.5, value=INIT_CUTOFF, description='Частота зрізу')
ch_noise = widgets.Checkbox(value=True, description='Показати шум')
btn_reset = widgets.Button(description='Reset', button_style='warning', icon='refresh')

# Створюємо спеціальний контейнер для залізобетонної фіксації ОДНОГО графіка
output_container = widgets.Output()

# 3. Головна функція відмальовування
def update_plot(*args):
    global current_noise, prev_mean, prev_cov
    
    # Використовуємо контейнер, щоб очистити все, що було намальовано всередині нього раніше
    with output_container:
        output_container.clear_output(wait=True)
        
        amp = s_amp.value
        freq = s_freq.value
        phase = s_phase.value
        n_mean = s_mean.value
        n_cov = s_cov.value
        cutoff = s_cutoff.value
        show_n = ch_noise.value
        
        # Перевірка умови ТЗ
        if n_mean != prev_mean or n_cov != prev_cov:
            current_noise = (base_noise * np.sqrt(n_cov)) + n_mean
            prev_mean = n_mean
            prev_cov = n_cov
            
        # Розрахунок сигналів
        pure_signal = amp * np.sin(2 * np.pi * freq * t + phase)
        noisy_signal = pure_signal + current_noise
        
        # Фільтрація Баттерворта
        nyq = 500.0
        normal_cutoff = cutoff / nyq
        b, a = butter(3, normal_cutoff, btype='low', analog=False)
        
        if show_n:
            filtered_signal = filtfilt(b, a, noisy_signal)
        else:
            filtered_signal = filtfilt(b, a, pure_signal)
            
        fig, (ax_orig, ax_filt) = plt.subplots(2, 1, figsize=(10, 6))
        
        # Верхній графік
        ax_orig.plot(t, pure_signal, label='Чиста гармоніка', color='green', lw=2)
        if show_n:
            ax_orig.plot(t, noisy_signal, label='Зашумлена гармоніка', color='orange', alpha=0.6)
        ax_orig.set_title("Початковий та зашумлений сигнал")
        ax_orig.legend(loc='upper right')
        ax_orig.set_xlim(0, 1)
        ax_orig.grid(True)
        
        # Нижній графік
        ax_filt.plot(t, pure_signal, label='Еталон (Чиста)', color='green', lw=1, linestyle='--')
        ax_filt.plot(t, filtered_signal, label='Відфільтрований сигнал (Butterworth)', color='purple', lw=2)
        ax_filt.set_title("Результат фільтрації (Низькочастотний Butterworth Фільтр)")
        ax_filt.legend(loc='upper right')
        ax_filt.set_xlim(0, 1)
        ax_filt.grid(True)
        
        plt.tight_layout()
        plt.show()

        print("\n ІНСТРУКЦІЯ КОРИСТУВАЧА:")
        print("1. Зміна лівих слайдерів (A, f, phi) не перегенерує масив випадкового шуму.")
        print("2. Слайдер 'Частота зрізу' адаптує лінійне згладжування низьких частот.")
        print("3. Чекбокс повністю вимикає зашумлену лінію на екрані.")
        print("4. Кнопка 'Reset' миттєво повертає всі слайдери до початкових значень ТЗ.")

# 4. Обробник події для кнопки Reset
def reset_values(b):
    s_amp.value = INIT_AMP
    s_freq.value = INIT_FREQ
    s_phase.value = INIT_PHASE
    s_mean.value = INIT_MEAN
    s_cov.value = INIT_COV
    s_cutoff.value = INIT_CUTOFF
    ch_noise.value = True

btn_reset.on_click(reset_values)

# 5. Прив'язка динамічного оновлення до кожного віджета
s_amp.observe(update_plot, names='value')
s_freq.observe(update_plot, names='value')
s_phase.observe(update_plot, names='value')
s_mean.observe(update_plot, names='value')
s_cov.observe(update_plot, names='value')
s_cutoff.observe(update_plot, names='value')
ch_noise.observe(update_plot, names='value')

# 6. Верстка та послідовне виведення елементів
controls = widgets.VBox([
    widgets.HBox([s_amp, s_mean]),
    widgets.HBox([s_freq, s_cov]),
    widgets.HBox([s_phase, s_cutoff]),
    widgets.HBox([ch_noise, btn_reset])
])

# Спочатку виводимо пульти, а під ними ОДИН чистий екран для графіків
display(controls)
display(output_container)

# Перший примусовий запуск
update_plot()

Output()